# TASHA activity-generation distributions (frequency, start time, duration, joint episodes, location bands)

Regenerate the empirical TASHA distributions from the estimation half of the RTS and compare them with the files used by the 2024 runs.

Part of DAYMAP (Daily Activity Model for Agent Plans), the activity-based model of the MILES platform, built on the ABIT code base. Not a Biogeme model: empirical distributions of the activity generator, built from the estimation half of the survey.

1. Raw data: survey files, half split
2. Data preparation: trips to typed activity episodes, building the distributions, rebuild check
3. Files
4. Descriptive statistics
5. The distributions

In [ ]:
RUN_DATA_PREP = True    # rebuild the split and the distributions into output/ (about 30 s) and compare with the shipped files
RUN_BUILD = RUN_DATA_PREP
import os, sys, json, glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt
FOLDER = os.path.abspath("")
sys.path.insert(0, os.path.join(FOLDER, "..", "..", "tools")); import nbtools as nt
RUN = json.load(open(os.path.join(FOLDER, "RUN.json")))
REF = os.path.join(FOLDER, "data", "reference_outputs")
OUT = os.path.join(FOLDER, "output"); os.makedirs(OUT, exist_ok=True)
pd.set_option("display.max_rows", 200, "display.max_columns", 60, "display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 1. Raw data

### 1.1 The 2017/18 Regional Travel Survey

All DAYMAP models are estimated on the 2017/18 RTS of the Baltimore and Washington regions (weekday travel day). Three files
are shipped in `../00_shared_data/rts_2017_18/raw`:

- `RTS_raw_flat_data.csv.gz`: the survey as received, one row per trip with its household, person and vehicle fields;
- `rts_trips_clean.csv.gz`: the trips after the survey cleaning (one row per trip, departure / arrival in minutes);
- `rts_tours.csv.gz`: home-based tours, work-based subtours and open tours built from the clean trips.

The cleaning script of the last two files is not shipped; the cleaned files are the starting point of the preparation here.
Race and ethnicity fields are not read.

In [ ]:
flat, trips, tours = nt.rts_raw()
split = pd.read_csv(os.path.join(nt.ABIT_SHARED, "rts_2017_18", "split", "rts_split.csv"))
display(nt.rts_counts(flat, trips, tours, split))
show = ["tripid", "household_id", "person_id", "tripno", "o_activity", "d_activity", "dep_min", "arr_min", "travel_mode",
        "distance", "hhsize", "numvehicle", "hh_income_broad", "age", "employment_status", "student_status", "wttrdfin"]
trips[[c for c in show if c in trips]].head()

### 1.2 Estimation and hold-out halves

The households are split in two halves (`make_rts_split.py`): strata of home county x household size (1 to 5+), a seeded
shuffle inside each stratum (seed 2026), then alternate assignment. Every model is estimated on the estimation half; the
hold-out (validation) half is the other half of the same survey and is not read by the estimation. With `RUN_DATA_PREP` the
split is rebuilt into a scratch folder and compared with the shipped `rts_split.csv`.

In [ ]:
REBUILD = os.path.join(OUT, "rebuild_check")            # scratch folder: provided files are never overwritten
if RUN_DATA_PREP:
    rc = nt.run_quiet(f"{sys.executable} make_rts_split.py", cwd=nt.ABIT_PREP, env={"ABIT_REBUILT": REBUILD})
    display(nt.compare_frames(pd.read_csv(os.path.join(REBUILD, "rts_2017_18", "split", "rts_split.csv")), split,
                              key=["household_id"], name="rts_split.csv"))
else:
    print("skipped (RUN_DATA_PREP = False)")
hh = flat.drop_duplicates("household_id").set_index("household_id")
hh["half"] = split.set_index("household_id")["half"]
t = pd.crosstab(hh["hhsize"].clip(upper=5).rename("household size (5 = 5+)"), hh["half"])
t.loc["all"] = t.sum()
t

## 2. Data preparation

### 2.1 From trips to activity episodes

`code/tasha_distributions.py` reads the trips of the estimation half and makes one episode per trip destination
(start = arrival, duration = departure of the next trip minus arrival; unknown for the last trip of the day). The destination
activity gives the episode type: work at the usual work tract before / after 15:00 (PW / SW), other work (WB), work at home
(WAHB), return home between two work episodes (RHW), school (SCH), shopping, recreation, accompany and other, each joint
(same household, purpose and tract, arrivals within 15 min, reported by 2+ members) or individual. The functions of the script
are called here as they are.

In [ ]:
sys.path.insert(0, os.path.join(FOLDER, "code")); import tasha_distributions as T
pm, per, t_est = T.load()
hh_child = pm.groupby("household_id").age.min().lt(16)
pm, wshares = T.segments(pm, per, hh_child)
e = T.episodes(t_est, per)
pseg = pm[~pm.imputed].drop_duplicates("person_id").set_index("person_id")
f = nt.Funnel("trips of the survey (clean file)", len(trips))
f.add("trips of the estimation half", len(t_est), "households of the hold-out half removed")
e = e[e.person_id.isin(pseg.index)].copy()
f.add("episodes of persons in the survey population", len(e), "person not in persons_map (no segment)")
f.add("out-of-home episodes", int((~e.is_home).sum()), "home episodes (only RHW is a type)")
e = T.subtours(e)
rt = e.wbs_n.gt(0); e.loc[rt, "dur"] = e.loc[rt, "merged_dur"]; e.loc[e.wbs | e.wbs_cont, "type"] = None
ev = e[e.type.notna()].copy()
f.add("typed episodes", len(ev), "stops of work-based outings and returns to the workplace are generated inside the work episode")
display(f.table())
print(f"persons of the estimation half: {pm.id.nunique():,} ({pm.imputed.sum():,} rows imputed for members without trips)")

In [ ]:
ev["seg"] = ev.person_id.map(pseg.seg); ev["group"] = ev.seg.map(T.GROUP)
ty = ev.groupby("type").agg(episodes=("type", "size"), weighted=("wttrdfin", "sum")).reindex(T.TYPES)
ty["share %"] = (100 * ty.episodes / ty.episodes.sum()).round(2); ty["weighted share %"] = (100 * ty.weighted / ty.weighted.sum()).round(2)
ty["purpose"] = [T.PURPOSE_OF[k] for k in ty.index]
ty.drop(columns="weighted")

### 2.2 Building the distributions

The script groups the episodes into cells (type x person segment x household class x episodes earlier in the day) and writes
one probability vector per cell with at least 50 (frequency) or 40 (start, duration, location) records; smaller cells fall
back to coarser ones. With `RUN_BUILD = True` it is run as shipped into `output/` and compared with the files used by the
model runs (`data/reference_outputs`).

In [ ]:
if RUN_BUILD:
    nt.run('PYTHONDONTWRITEBYTECODE=1 TASHA_JOINT_PB=1 TASHA_LIC_TYPES=IO_ACC,IS,IO_REC,IO_OTH python3 tasha_distributions.py', cwd=os.path.join(FOLDER, "code"))
else:
    print("skipped (RUN_BUILD = False)")

In [ ]:
if RUN_BUILD:
    nt.run('python3 compare_with_reference.py', cwd=os.path.join(FOLDER, "code"))
else:
    print("skipped (RUN_BUILD = False)")

Three files differ from the reference by design: the joint participants and propensities were kept from an earlier build
(before work-based outings were removed from the episodes; differences of a few episodes), and `tasha_params.csv` of the
reference also holds engine settings appended when the coefficient files were assembled.

## 3. Files

`data/reference_outputs` holds the files used by the model runs; `output/` the rebuild.

In [ ]:
display(nt.files_table(REF))
src = OUT if glob.glob(os.path.join(OUT, "tasha_*.csv")) else REF
print("charts of section 5 from:", os.path.relpath(src, FOLDER))
def probs(t):
    return np.vstack(t["probs"].map(lambda s: np.array([float(x) for x in str(s).split(";")])).to_numpy())

## 4. Descriptive statistics

### Persons and households of the estimation half
Person segments (weighted by the person weight `w`), household size and age.

In [ ]:
P = pm.drop_duplicates("id")
fig, ax = nt.panels(3, ncol=3, w=4.4, h=3)
s = (P.groupby("seg").w.sum() / P.w.sum() * 100).reindex(T.SEGS)
nt.bar(s, "person segment", "weighted share of persons (%)", ax=ax[0], rot=45)
H = P.drop_duplicates("household_id")
nt.bar(H.hhsize.clip(upper=6).value_counts().sort_index(), "household size (6 = 6+)", "households", ax=ax[1])
nt.hist(P.age, "age (years)", "persons", bins=range(0, 96, 5), ax=ax[2])
plt.tight_layout(); plt.show()
w = P.w
print(f"person weight w: mean {w.mean():.1f}, median {w.median():.1f}, min {w.min():.1f}, max {w.max():.1f}; sum {w.sum():,.0f} persons")

### Episodes by start hour and duration
Arrival hour and duration of the typed episodes by group of types (unweighted).

In [ ]:
grp = {"work": ["WB", "WAHB", "PW", "SW", "RHW"], "school": ["SCH"], "shopping": ["JS", "IS"],
       "recreation / other": ["JO_REC", "JO_OTH", "IO_REC", "IO_OTH"], "accompany": ["IO_ACC"]}
fig, ax = nt.panels(len(grp), ncol=5, w=3.1, h=2.6)
for a, (g, tys) in zip(ax, grp.items()):
    x = ev[ev.type.isin(tys)]
    a.hist(x.arr_min / 60, bins=np.arange(0, 24.5, 0.5), color=nt.OBS); a.set_xlabel(f"arrival hour ({g})"); a.set_xticks([0, 6, 12, 18, 24])
ax[0].set_ylabel("episodes"); plt.tight_layout(); plt.show()
fig, ax = nt.panels(len(grp), ncol=5, w=3.1, h=2.6)
for a, (g, tys) in zip(ax, grp.items()):
    x = ev[ev.type.isin(tys) & ev.dur.notna()]
    a.hist(x.dur / 60, bins=np.arange(0, 12.25, 0.25), color=nt.OBS); a.set_xlabel(f"duration, h ({g})")
ax[0].set_ylabel("episodes"); plt.tight_layout(); plt.show()

### Estimation half vs hold-out half
Trip purposes (destination activity, weighted by the trip weight) in the two halves.

In [ ]:
half = trips.household_id.map(split.set_index("household_id")["half"])
pur = trips.d_activity.map(T.ACT_PURPOSE).fillna("OTHER")
display(nt.half_comparison(pur[half.eq("estimation")], pur[half.eq("validation")], "destination purpose",
                           trips.wttrdfin[half.eq("estimation")], trips.wttrdfin[half.eq("validation")]))

## 5. The distributions

### Frequency P(n | type, segment, household, prior): mean episodes per person-day

In [ ]:
fq = pd.read_csv(os.path.join(src, "tasha_frequency.csv"))
fq = fq[fq["level"] == fq["level"].iloc[0]] if "level" in fq else fq
fq = fq[fq["seg"].isin(T.SEGS)]
m = fq.groupby(["type", "seg"])[["mean", "n"]].apply(lambda g: np.average(g["mean"], weights=g["n"])).unstack().reindex(columns=[c for c in T.SEGS if c in set(fq["seg"])])
fig, ax = plt.subplots(figsize=(min(14, 1.0 * m.shape[1] + 3), 0.42 * m.shape[0] + 1.5))
im = ax.imshow(m.values, cmap="Greys"); plt.colorbar(im, ax=ax, label="mean episodes per person-day (weighted by cell size)")
ax.set_xticks(range(m.shape[1])); ax.set_xticklabels(m.columns, rotation=45, ha="right"); ax.set_yticks(range(m.shape[0])); ax.set_yticklabels(m.index)
ax.set_xlabel("person segment"); ax.set_ylabel("episode type")
for i in range(m.shape[0]):
    for j in range(m.shape[1]):
        if np.isfinite(m.values[i, j]): ax.text(j, i, f"{m.values[i, j]:.2f}", ha="center", va="center", fontsize=7, color="white" if m.values[i, j] > np.nanmax(m.values) * 0.6 else "black")
plt.tight_layout(); plt.show()

### Start time P(t | type, …): 15-minute bins from midnight, averaged over the cells of each type

In [ ]:
stt = pd.read_csv(os.path.join(src, "tasha_start.csv"))
P_ = probs(stt); hours = np.arange(P_.shape[1]) * 0.25
types = sorted(stt["type"].unique())
fig, axes = nt.panels(len(types), ncol=4, w=3.7, h=2.3)
for ax, t_ in zip(axes, types):
    i = (stt["type"] == t_).to_numpy(); w_ = stt.loc[i, "n"].to_numpy()
    ax.fill_between(hours, 100 * (P_[i] * w_[:, None]).sum(0) / w_.sum(), step="mid", color=nt.OBS, alpha=0.85)
    ax.set_xlabel(f"start hour ({t_})"); ax.set_xticks([0, 6, 12, 18, 24])
fig.supylabel("probability per 15 min (%)"); plt.tight_layout(); plt.show()

### Duration P(δ | type, …): 5-minute bins to 2 h, then 15 minutes

In [ ]:
du = pd.read_csv(os.path.join(src, "tasha_duration.csv"))
edges = np.array(list(range(0, 121, 5)) + list(range(135, 1441, 15)), float); mids = (edges[:-1] + edges[1:]) / 2 / 60
P_ = probs(du)[:, :len(mids)]
types = sorted(du["type"].unique())
fig, axes = nt.panels(len(types), ncol=4, w=3.7, h=2.3)
for ax, t_ in zip(axes, types):
    i = (du["type"] == t_).to_numpy(); w_ = du.loc[i, "n"].to_numpy()
    dens = (P_[i] * w_[:, None]).sum(0) / w_.sum() / np.diff(edges) * 60
    ax.plot(mids, dens, color=nt.OBS); ax.set_xlim(0, 12); ax.set_xlabel(f"duration, h ({t_})")
fig.supylabel("density per hour"); plt.tight_layout(); plt.show()

### Joint episodes, location bands and scheduling parameters

In [ ]:
for f_ in ["tasha_joint_propensity.csv", "tasha_joint_propensity_mand.csv", "tasha_joint_participants.csv", "tasha_location_bands.csv", "tasha_params.csv"]:
    p = os.path.join(src, f_)
    if os.path.exists(p):
        print(f_); display(pd.read_csv(p))